# 05 — Final Merge, EDA, Preprocessing, Feature Engineering & Train/Test Split

**Owner:** Member D  
**Depends on:** Notebooks 01–04 must be run first.  
**Inputs:** `data/processed/` — all 7 cleaned datasets  
**Outputs:** `data/final/` — X_train, X_val, X_test, y_train, y_val, y_test, scaler.pkl, feature_names.txt

### Scope
This notebook merges every processed dataset into a single order-level feature table,
performs post-merge EDA (Stage 3), applies final preprocessing and feature engineering (Stage 4),
runs a leakage audit, splits chronologically, scales, and saves modelling-ready outputs.

`sellers_geo_clean` and `geolocation_zip_clean` are loaded for audit only — their geographic
information already flows through `customers_geo_clean` and `order_geo_features` respectively.


## 1. Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.preprocessing import StandardScaler

# ── Paths ──────────────────────────────────────────────────────────────────
REPO_ROOT   = Path.cwd().parent
PROCESSED   = REPO_ROOT / 'data' / 'processed'
FINAL_DIR   = REPO_ROOT / 'data' / 'final'
FIGURES_DIR = REPO_ROOT / 'report' / 'figures' / 'member_d'
FINAL_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Plot style ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', context='notebook')
PRIMARY   = '#287271'
SECONDARY = '#D1495B'
ACCENT    = '#F4A261'
plt.rcParams['figure.dpi'] = 110

# ── Leakage guard ──────────────────────────────────────────────────────────
PROHIBITED = (
    'is_late_delivery', 'delivered_customer', 'delivered_carrier',
    'delivery_duration', 'late_days', 'review', 'cancellation',
    'order_status', 'min_shipping_limit_date', 'max_shipping_limit_date',
)

# ── Chronological split percentiles ────────────────────────────────────────
TRAIN_CUTOFF_PCTILE = 0.70
VAL_CUTOFF_PCTILE   = 0.85

print('Repo root :', REPO_ROOT)
print('Processed :', PROCESSED)
print('Final out :', FINAL_DIR)


## 2. Load All 7 Processed Datasets & Pre-Merge Audit

All 7 files from `data/processed/` are loaded and audited before any merge.

| Dataset | Role | Join Key |
|---------|------|----------|
| `orders_clean` | **Spine** (has target + `customer_id`) | `order_id` |
| `customers_geo_clean` | Direct join via `customer_id` | `customer_id` |
| `order_geo_features` | Direct join via `order_id` | `order_id` |
| `payments_clean` | Direct join via `order_id` | `order_id` |
| `products_items_clean` | Direct join via `order_id` | `order_id` |
| `sellers_geo_clean` | **Audit-only** — seller_id has no order-level bridge (already aggregated in order_geo_features) | `seller_id` |
| `geolocation_zip_clean` | **Audit-only** — lat/lng already in customers_geo_clean; diagnostic cols not useful for ML | `zip_prefix` |


In [ ]:
# ── 2.1 Load ───────────────────────────────────────────────────────────────
orders = pd.read_csv(
    PROCESSED / 'orders_clean.csv',
    dtype={'order_id': 'string', 'customer_id': 'string'},
    parse_dates=['order_purchase_timestamp'],
)

customers_geo = pd.read_csv(
    PROCESSED / 'customers_geo_clean.csv',
    dtype={
        'customer_id': 'string', 'customer_unique_id': 'string',
        'customer_zip_code_prefix': 'string',
        'customer_city': 'string', 'customer_state': 'string',
        'customer_region': 'string',
    },
)

order_geo = pd.read_csv(
    PROCESSED / 'order_geo_features.csv',
    dtype={
        'order_id': 'string',
        'customer_state': 'string', 'customer_region': 'string',
        'any_seller_same_state': 'boolean', 'all_sellers_same_state': 'boolean',
        'any_seller_same_region': 'boolean', 'all_sellers_same_region': 'boolean',
        'missing_customer_coordinates': 'boolean',
        'missing_seller_coordinates': 'boolean',
    },
)

payments = pd.read_csv(
    PROCESSED / 'payments_clean.csv',
    dtype={
        'order_id': 'string', 'primary_payment_type': 'string',
        'has_boleto': 'boolean', 'has_credit_card': 'boolean',
        'has_debit_card': 'boolean', 'has_other': 'boolean',
        'has_voucher': 'boolean', 'is_high_value_order': 'boolean',
    },
)

products_items = pd.read_csv(
    PROCESSED / 'products_items_clean.csv',
    dtype={'order_id': 'string', 'dominant_product_category': 'string'},
    parse_dates=['min_shipping_limit_date', 'max_shipping_limit_date'],
)

# Audit-only datasets (not merged into spine)
sellers_geo = pd.read_csv(PROCESSED / 'sellers_geo_clean.csv',
                          dtype={'seller_id': 'string'})
geo_zip     = pd.read_csv(PROCESSED / 'geolocation_zip_clean.csv',
                          dtype={'geolocation_zip_code_prefix': 'string'})

print('All 7 datasets loaded.')


In [ ]:
# ── 2.2 Pre-merge audit ────────────────────────────────────────────────────
audit_specs = [
    ('orders_clean',          orders,         'order_id',                    'join'),
    ('customers_geo_clean',   customers_geo,  'customer_id',                 'join'),
    ('order_geo_features',    order_geo,      'order_id',                    'join'),
    ('payments_clean',        payments,       'order_id',                    'join'),
    ('products_items_clean',  products_items, 'order_id',                    'join'),
    ('sellers_geo_clean',     sellers_geo,    'seller_id',                   'audit-only'),
    ('geolocation_zip_clean', geo_zip,        'geolocation_zip_code_prefix', 'audit-only'),
]

audit_rows = []
for name, df, key, role in audit_specs:
    audit_rows.append({
        'dataset'        : name,
        'role'           : role,
        'rows'           : len(df),
        'columns'        : df.shape[1],
        'key'            : key,
        'key_unique'     : df[key].is_unique,
        'key_null_count' : int(df[key].isna().sum()),
        'total_nulls'    : int(df.isna().sum().sum()),
        'null_pct'       : round(df.isna().mean().mean() * 100, 2),
    })

audit_df = pd.DataFrame(audit_rows).set_index('dataset')
display(Markdown('### 2.2 Pre-Merge Dataset Audit'), audit_df)


**Interpretation.** All 7 datasets are loaded. Keys are unique and non-null in every table.
`sellers_geo_clean` (key: `seller_id`) cannot be joined to the order-level spine without a raw
`order_items` bridge — but this aggregation was already performed in Notebook 02, producing
`order_geo_features`. `geolocation_zip_clean` is a ZIP-level lookup whose lat/lng is already
embedded in `customers_geo_clean`; its diagnostic columns are not predictive features.


## 3. Five-Step Validated Merge

The spine is `orders_clean` (96,470 rows). Each join uses `how='left'` and `validate='1:1'`
to preserve row count and immediately raise errors on unexpected fan-outs.

`customers_geo_clean` supplies `customer_state` and `customer_region` too, but those columns
are **dropped in favour of the versions from `order_geo_features`** (verified at order level
in Notebook 02).


In [ ]:
# ── Step 1: spine ──────────────────────────────────────────────────────────
merged = orders.copy()
print(f'Step 0  — spine (orders_clean):          {len(merged):>7,} rows')

# ── Step 2: customers_geo via customer_id ──────────────────────────────────
# Select only columns NOT duplicated in order_geo_features to avoid _x/_y conflicts
cust_join_cols = [
    'customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
    'customer_city', 'customer_latitude', 'customer_longitude',
    'customer_geolocation_missing',
]
merged = merged.merge(
    customers_geo[cust_join_cols],
    on='customer_id', how='left', validate='1:1',
)
merged.drop(columns=['customer_id'], inplace=True)   # bridge key, not a feature
print(f'Step 1  — + customers_geo_clean:         {len(merged):>7,} rows  (+{len(cust_join_cols)-1} cols)')

# ── Step 3: order_geo_features via order_id ────────────────────────────────
merged = merged.merge(order_geo, on='order_id', how='left', validate='1:1')
print(f'Step 2  — + order_geo_features:          {len(merged):>7,} rows')

# ── Step 4: payments via order_id ─────────────────────────────────────────
merged = merged.merge(payments, on='order_id', how='left', validate='1:1')
print(f'Step 3  — + payments_clean:              {len(merged):>7,} rows')

# ── Step 5: products_items via order_id ───────────────────────────────────
merged = merged.merge(products_items, on='order_id', how='left', validate='1:1')
print(f'Step 4  — + products_items_clean:        {len(merged):>7,} rows')

# ── Sanity checks ─────────────────────────────────────────────────────────
assert len(merged) == len(orders), 'Row count changed — left join broken!'
suffix_cols = [c for c in merged.columns if c.endswith(('_x', '_y'))]
assert not suffix_cols, f'Unexpected suffix columns: {suffix_cols}'
print(f'\nFinal shape : {merged.shape}')
print(f'Suffix cols : none  ✔')
print(f'Row count   : preserved ({len(merged):,})  ✔')

# Save merged raw dataset before split
MERGED_OUT = PROCESSED / 'merged_raw_orders.csv'
merged.to_csv(MERGED_OUT, index=False)
print(f'Saved merged raw dataset to {MERGED_OUT}')


In [ ]:
# ── 3.1 Full schema view ───────────────────────────────────────────────────
def get_source(col):
    if col in orders.columns:            return 'orders'
    if col in customers_geo.columns:     return 'customers_geo'
    if col in order_geo.columns:         return 'order_geo'
    if col in payments.columns:          return 'payments'
    return 'products_items'

schema = pd.DataFrame({
    'dtype'      : merged.dtypes.astype(str),
    'null_count' : merged.isna().sum(),
    'null_pct'   : (merged.isna().mean() * 100).round(2),
    'source'     : [get_source(c) for c in merged.columns],
})
display(Markdown('### 3.1 Merged Dataset Schema'), schema)


## 4. Post-Merge EDA

All EDA uses the full merged dataset. The target is described only — it is not used to drive any preprocessing decision.

### 4.1 Target Distribution & Class Imbalance

In [ ]:
target_vc = merged['is_late_delivery'].value_counts().sort_index()
labels = ['On-time (0)', 'Late (1)']
colors = [PRIMARY, SECONDARY]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bars = axes[0].bar(labels, target_vc.values, color=colors, edgecolor='#222')
for bar, val in zip(bars, target_vc.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 400,
                 f'{val:,}\n({val/len(merged)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Target Class Counts', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of orders')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

axes[1].pie(target_vc.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': '#222', 'linewidth': 1.2})
axes[1].set_title('Target Class Proportions', fontsize=13, fontweight='bold')

plt.suptitle('is_late_delivery Distribution  (n = {:,})'.format(len(merged)),
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'target_distribution.png', bbox_inches='tight')
plt.show()

print(f"Class imbalance ratio  : {target_vc[0]/target_vc[1]:.2f}:1  (on-time : late)")
print(f"Late-delivery rate     : {target_vc[1]/len(merged)*100:.2f}%")


**Observation.** The target is heavily imbalanced: **~91.9% on-time vs 8.1% late** (ratio ≈ 11.3:1). This must be addressed during modelling (class weights, SMOTE, threshold tuning) but is not handled here.

### 4.2 Missing-Value Summary (Post-Merge)

In [ ]:
miss = pd.DataFrame({
    'missing_count' : merged.isna().sum(),
    'missing_pct'   : (merged.isna().mean() * 100).round(3),
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

display(Markdown('#### Columns with missing values'), miss)

if len(miss) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(miss) * 0.45)))
    bars = ax.barh(miss.index[::-1], miss['missing_pct'][::-1], color=SECONDARY, edgecolor='#222')
    for bar, pct in zip(bars, miss['missing_pct'][::-1]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f'{pct:.2f}%', va='center', fontsize=9)
    ax.set_xlabel('Missing %')
    ax.set_title('Post-Merge Missing Value Rates', fontweight='bold')
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'missing_values_postmerge.png', bbox_inches='tight')
    plt.show()
else:
    print('No missing values found in merged dataset.')


**Observation.** Missing values fall into three groups: (1) distance/geo flag columns (~0.8–1.3%) from orders with no seller bridge; (2) product columns (~0.8%) from orders absent in the order-items table; (3) `shipping_limit_lead_days` derived from dates. All will be imputed in Section 6.

### 4.3 Late-Delivery Rate by Key Categorical Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# By customer region
region_late = merged.groupby('customer_region')['is_late_delivery'].mean().sort_values()
axes[0,0].barh(region_late.index, region_late.values * 100, color=PRIMARY, edgecolor='#222')
axes[0,0].set_xlabel('Late delivery rate (%)')
axes[0,0].set_title('Late Rate by Customer Region', fontweight='bold')
for i, (idx, val) in enumerate(region_late.items()):
    axes[0,0].text(val*100 + 0.1, i, f'{val*100:.1f}%', va='center', fontsize=9)

# By purchase month
month_late = merged.groupby('purchase_month')['is_late_delivery'].mean()
axes[0,1].plot(month_late.index, month_late.values * 100, marker='o',
               color=PRIMARY, linewidth=2, markersize=6)
axes[0,1].fill_between(month_late.index, month_late.values * 100, alpha=0.15, color=PRIMARY)
axes[0,1].set_xlabel('Purchase month')
axes[0,1].set_ylabel('Late delivery rate (%)')
axes[0,1].set_title('Late Rate by Purchase Month', fontweight='bold')
axes[0,1].set_xticks(range(1, 13))
axes[0,1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                            'Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)

# By day of week
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_late = merged.groupby('purchase_dayofweek')['is_late_delivery'].mean()
axes[1,0].bar(dow_late.index, dow_late.values * 100, color=ACCENT, edgecolor='#222')
axes[1,0].set_xticks(range(7))
axes[1,0].set_xticklabels(dow_labels)
axes[1,0].set_ylabel('Late delivery rate (%)')
axes[1,0].set_title('Late Rate by Day of Week', fontweight='bold')

# By primary payment type
pay_late = merged.groupby('primary_payment_type')['is_late_delivery'].mean().sort_values()
axes[1,1].barh(pay_late.index, pay_late.values * 100, color=SECONDARY, edgecolor='#222')
axes[1,1].set_xlabel('Late delivery rate (%)')
axes[1,1].set_title('Late Rate by Payment Type', fontweight='bold')
for i, (idx, val) in enumerate(pay_late.items()):
    axes[1,1].text(val*100 + 0.05, i, f'{val*100:.1f}%', va='center', fontsize=9)

plt.suptitle('Late-Delivery Rate by Key Categorical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'late_rate_by_categorical.png', bbox_inches='tight')
plt.show()


### 4.4 Numeric Feature Distributions by Target Class

In [ ]:
numeric_focus = [
    'estimated_delivery_days', 'approval_lag_hours',
    'average_distance_km', 'total_price',
    'total_freight_value', 'total_weight_g',
    'item_count', 'max_installments',
]
numeric_focus = [c for c in numeric_focus if c in merged.columns]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, col in zip(axes, numeric_focus):
    for val, label, color in [(0, 'On-time', PRIMARY), (1, 'Late', SECONDARY)]:
        subset = merged.loc[merged['is_late_delivery'] == val, col].dropna()
        ax.hist(subset, bins=40, alpha=0.55, color=color, label=label, density=True)
    ax.set_title(col.replace('_', ' ').title(), fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)

plt.suptitle('Numeric Feature Distributions by Target Class', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'numeric_distributions_by_target.png', bbox_inches='tight')
plt.show()


### 4.5 Correlation Heatmap (Numeric Features)

In [ ]:
bool_cols_all = merged.select_dtypes(include='boolean').columns.tolist()
corr_df = merged.copy()
for c in bool_cols_all:
    corr_df[c] = corr_df[c].astype('Int64')

numeric_cols = corr_df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix  = corr_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, ax=ax,
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 6},
    linewidths=0.3, square=True,
)
ax.set_title('Pearson Correlation Matrix (Numeric Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'correlation_heatmap.png', bbox_inches='tight')
plt.show()

target_corr = corr_matrix['is_late_delivery'].drop('is_late_delivery').abs().sort_values(ascending=False)
display(Markdown('#### Top 15 correlations with `is_late_delivery`'),
        target_corr.head(15).to_frame('|corr|'))


### 4.6 Geographic Distance vs Late Delivery

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

data_0 = merged.loc[merged['is_late_delivery']==0, 'average_distance_km'].dropna()
data_1 = merged.loc[merged['is_late_delivery']==1, 'average_distance_km'].dropna()
axes[0].boxplot([data_0, data_1], tick_labels=['On-time', 'Late'],
                patch_artist=True,
                boxprops=dict(facecolor=PRIMARY, alpha=0.7),
                medianprops=dict(color=SECONDARY, linewidth=2))
axes[0].set_ylabel('Average distance (km)')
axes[0].set_title('Customer-Seller Distance by Delivery Outcome', fontweight='bold')

merged_dist = merged.dropna(subset=['average_distance_km']).copy()
merged_dist['dist_decile'] = pd.qcut(merged_dist['average_distance_km'], 10, labels=False)
decile_late      = merged_dist.groupby('dist_decile')['is_late_delivery'].mean() * 100
decile_mean_dist = merged_dist.groupby('dist_decile')['average_distance_km'].mean()
axes[1].bar(range(10), decile_late.values, color=PRIMARY, edgecolor='#222')
axes[1].set_xticks(range(10))
axes[1].set_xticklabels([f'{int(d):,}' for d in decile_mean_dist.values], rotation=45, fontsize=8)
axes[1].set_xlabel('Mean distance in decile (km)')
axes[1].set_ylabel('Late delivery rate (%)')
axes[1].set_title('Late Rate by Distance Decile', fontweight='bold')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'distance_vs_late_delivery.png', bbox_inches='tight')
plt.show()


## 5. Leakage Audit

The prediction point is **after order confirmation, before delivery**.
Any column derived from post-delivery events is leakage and must not enter the feature set.


In [ ]:
leakage_rows = []
for col in merged.columns:
    is_prohibited = any(frag in col.lower() for frag in PROHIBITED)
    leakage_rows.append({
        'column'     : col,
        'dtype'      : str(merged[col].dtype),
        'prohibited' : is_prohibited,
        'action'     : 'DROP (leakage / date bridge)' if is_prohibited else 'KEEP',
    })

leakage_df = pd.DataFrame(leakage_rows).set_index('column')
flagged    = leakage_df[leakage_df['prohibited']]
display(Markdown('### 5.1 Flagged Columns'), flagged)
display(Markdown('### 5.2 Full Column Audit'), leakage_df)


In [ ]:
# Columns to drop at feature-set stage (documented)
DROP_BEFORE_FEATURES = [
    'order_purchase_timestamp',   # used for split only
    'min_shipping_limit_date',    # raw date — safe feature derived below
    'max_shipping_limit_date',    # raw date — not needed after derivation
    'is_late_delivery',           # target — moved to y sets
]
print('Columns to drop before building X:')
for c in DROP_BEFORE_FEATURES:
    print(f'  {c:<38} present={c in merged.columns}')


**Interpretation.** `min_shipping_limit_date` and `max_shipping_limit_date` are seller-committed
dates set at purchase time — not leakage per se, but kept only long enough to derive
`shipping_limit_lead_days`. `order_purchase_timestamp` drives the chronological split and
is then removed. `is_late_delivery` becomes the y vector.


## 6. Final Preprocessing

### 6.1 Engineer `shipping_limit_lead_days` (Before Dropping Dates)

Days from purchase to earliest seller shipping commitment.
Known at order time; acts as a seller time-pressure proxy.


In [ ]:
if 'min_shipping_limit_date' in merged.columns:
    merged['shipping_limit_lead_days'] = (
        pd.to_datetime(merged['min_shipping_limit_date'])
        - merged['order_purchase_timestamp']
    ).dt.total_seconds() / 86_400
    print('shipping_limit_lead_days created.')
    print(merged['shipping_limit_lead_days'].describe().round(3))


### 6.2 Chronological Train / Val / Test Split

Split is done **before imputation** so that imputation statistics are
computed on train only, preventing any information from val/test leaking into train.

| Split | Cutoff | Approx. ratio |
|-------|--------|---------------|
| Train | ≤ 70th percentile timestamp | 70% |
| Val   | 70th–85th percentile | 15% |
| Test  | > 85th percentile | 15% |


In [ ]:
merged_sorted = merged.sort_values('order_purchase_timestamp').reset_index(drop=True)

train_cutoff = merged_sorted['order_purchase_timestamp'].quantile(TRAIN_CUTOFF_PCTILE)
val_cutoff   = merged_sorted['order_purchase_timestamp'].quantile(VAL_CUTOFF_PCTILE)

train_mask = merged_sorted['order_purchase_timestamp'] <= train_cutoff
val_mask   = (merged_sorted['order_purchase_timestamp'] > train_cutoff) & \
             (merged_sorted['order_purchase_timestamp'] <= val_cutoff)
test_mask  = merged_sorted['order_purchase_timestamp'] > val_cutoff

df_train = merged_sorted[train_mask].copy()
df_val   = merged_sorted[val_mask].copy()
df_test  = merged_sorted[test_mask].copy()

print('Split cutoffs:')
print(f'  Train <= {train_cutoff.date()}   ->  {len(df_train):>7,} rows  ({len(df_train)/len(merged)*100:.1f}%)')
print(f'  Val   <= {val_cutoff.date()}     ->  {len(df_val):>7,} rows  ({len(df_val)/len(merged)*100:.1f}%)')
print(f'  Test  >  {val_cutoff.date()}     ->  {len(df_test):>7,} rows  ({len(df_test)/len(merged)*100:.1f}%)')
assert len(df_train) + len(df_val) + len(df_test) == len(merged), 'Split row count mismatch!'

print()
for name, df_s in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    rate = df_s['is_late_delivery'].mean()
    print(f'  {name} late rate: {rate*100:.2f}%  (late count: {int(df_s["is_late_delivery"].sum()):,})')


In [ ]:
# Drop timestamp and raw date columns after split
for df_s in [df_train, df_val, df_test]:
    df_s.drop(columns=[c for c in ['order_purchase_timestamp',
                                    'min_shipping_limit_date',
                                    'max_shipping_limit_date'] if c in df_s.columns],
              inplace=True)
print('Timestamps and raw date columns dropped from all splits.')


### 6.3 Missing Value Imputation (fit on train, transform all splits)

In [ ]:
# ── Distance columns: train-median imputation ──────────────────────────────
dist_cols = ['minimum_distance_km', 'average_distance_km', 'maximum_distance_km']
dist_medians = {c: df_train[c].median() for c in dist_cols if c in df_train.columns}
for df_s in [df_train, df_val, df_test]:
    for col, med in dist_medians.items():
        df_s[col] = df_s[col].fillna(med)
print('Distance medians (from train):')
for col, med in dist_medians.items():
    print(f'  {col}: {med:.2f} km')

# ── Seller boolean flags: fill False (no seller bridge = unknown) ───────────
seller_flag_cols = [
    'any_seller_same_state', 'all_sellers_same_state',
    'any_seller_same_region', 'all_sellers_same_region',
    'missing_seller_coordinates',
]
for df_s in [df_train, df_val, df_test]:
    for col in seller_flag_cols:
        if col in df_s.columns:
            df_s[col] = df_s[col].fillna(False)

# ── Product numeric columns: fill 0 (no item-bridge record) ────────────────
prod_fill0 = [
    'item_count', 'distinct_seller_count', 'distinct_product_count',
    'distinct_category_count', 'total_price', 'average_item_price',
    'total_freight_value', 'average_freight_value',
    'total_weight_g', 'average_weight_g', 'total_volume_cm3',
    'missing_weight_count', 'missing_dims_count',
    'outlier_price_item_count', 'outlier_freight_item_count',
]
for df_s in [df_train, df_val, df_test]:
    for col in prod_fill0:
        if col in df_s.columns:
            df_s[col] = df_s[col].fillna(0)

# ── dominant_product_category: fill 'other' ────────────────────────────────
for df_s in [df_train, df_val, df_test]:
    if 'dominant_product_category' in df_s.columns:
        df_s['dominant_product_category'] = df_s['dominant_product_category'].fillna('other')

# ── shipping_limit_lead_days: train-median ─────────────────────────────────
if 'shipping_limit_lead_days' in df_train.columns:
    lead_med = df_train['shipping_limit_lead_days'].median()
    for df_s in [df_train, df_val, df_test]:
        df_s['shipping_limit_lead_days'] = df_s['shipping_limit_lead_days'].fillna(lead_med)
    print(f'shipping_limit_lead_days median (train): {lead_med:.2f} days')

print()
print('Remaining nulls after imputation:')
for name, df_s in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    n = df_s.isna().sum().sum()
    print(f'  {name}: {n} total nulls')


### 6.4 Boolean Columns → int8

In [ ]:
for df_s in [df_train, df_val, df_test]:
    bool_cols_in = df_s.select_dtypes(include='boolean').columns.tolist()
    for col in bool_cols_in:
        df_s[col] = df_s[col].astype('Int64').fillna(0).astype('int8')
remaining_bool = df_train.select_dtypes(include='boolean').columns.tolist()
print('Boolean columns converted to int8.')
print('Remaining boolean dtype columns:', remaining_bool if remaining_bool else 'none')


## 7. Cross-Dataset Feature Engineering

8 new features derived from the merged table that were not possible in individual notebooks.
All use information available at prediction time (post-purchase, pre-delivery).

| Feature | Formula | Rationale |
|---------|---------|-----------|
| `freight_to_price_ratio` | `total_freight / total_price` | Relative shipping burden |
| `weight_per_item` | `total_weight_g / item_count` | Average item heaviness |
| `volume_per_item` | `total_volume_cm3 / item_count` | Average item bulk |
| `is_multi_seller` | `distinct_seller_count > 1` | Multi-seller = coordination overhead |
| `is_cross_state` | `any_seller_same_state == 0` | No same-state seller = cross-state shipping |
| `distance_x_items` | `average_distance_km × item_count` | Compound logistics difficulty |
| `freight_per_km` | `total_freight / average_distance_km` | Logistics cost efficiency |
| `purchase_quarter` | `ceil(purchase_month / 3)` | Seasonal grouping |


In [ ]:
def engineer_features(df):
    df = df.copy()

    # 1. freight_to_price_ratio
    price_denom = df['total_price'].replace(0, np.nan)
    df['freight_to_price_ratio'] = (df['total_freight_value'] / price_denom).fillna(0)

    # 2. weight_per_item
    item_denom = df['item_count'].replace(0, np.nan)
    df['weight_per_item'] = (df['total_weight_g'] / item_denom).fillna(0)

    # 3. volume_per_item
    df['volume_per_item'] = (df['total_volume_cm3'] / item_denom).fillna(0)

    # 4. is_multi_seller
    df['is_multi_seller'] = (df['distinct_seller_count'] > 1).fillna(False).astype('int8')

    # 5. is_cross_state (no seller in same state as customer)
    df['is_cross_state'] = (df['any_seller_same_state'] == 0).fillna(False).astype('int8')

    # 6. distance_x_items (compound logistics difficulty)
    df['distance_x_items'] = df['average_distance_km'] * df['item_count']

    # 7. freight_per_km
    dist_denom = df['average_distance_km'].replace(0, np.nan)
    df['freight_per_km'] = (df['total_freight_value'] / dist_denom).fillna(0)

    # 8. purchase_quarter
    df['purchase_quarter'] = ((df['purchase_month'] - 1) // 3 + 1).astype('int8')

    return df

df_train = engineer_features(df_train)
df_val   = engineer_features(df_val)
df_test  = engineer_features(df_test)

new_feats = [
    'freight_to_price_ratio', 'weight_per_item', 'volume_per_item',
    'is_multi_seller', 'is_cross_state', 'distance_x_items',
    'freight_per_km', 'purchase_quarter',
]
print('Engineered features (train stats):')
for f in new_feats:
    print(f'  {f:<30} mean={df_train[f].mean():.4f}  std={df_train[f].std():.4f}')


In [ ]:
# ── Visualise engineered features vs target ────────────────────────────────
eng_vis = [
    'freight_to_price_ratio', 'weight_per_item', 'volume_per_item',
    'distance_x_items', 'freight_per_km', 'shipping_limit_lead_days',
]
eng_vis = [c for c in eng_vis if c in df_train.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for ax, col in zip(axes, eng_vis):
    for val, label, color in [(0, 'On-time', PRIMARY), (1, 'Late', SECONDARY)]:
        subset = df_train.loc[df_train['is_late_delivery'] == val, col].dropna()
        cap = subset.quantile(0.99)
        ax.hist(subset.clip(upper=cap), bins=40, alpha=0.55, color=color, label=label, density=True)
    ax.set_title(col.replace('_', ' ').title(), fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)

plt.suptitle('Engineered Feature Distributions by Target (Train Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'engineered_features_by_target.png', bbox_inches='tight')
plt.show()


## 8. Categorical Encoding

All encoding is **fit on train only** and applied identically to val and test to prevent leakage.

| Column | Method | Reason |
|--------|--------|--------|
| `customer_state` (27 cats) | Target-frequency encoding | High cardinality; avoids 27 dummies |
| `dominant_product_category` (22 cats) | Target-frequency encoding | Medium-high cardinality |
| `customer_region` (5 cats) | One-hot (drop first) | Low cardinality; interpretable |
| `primary_payment_type` (5 cats) | One-hot (drop first) | Low cardinality |


In [ ]:
# ── 8.1 Target-frequency encoding ─────────────────────────────────────────
TARGET_FREQ_COLS = ['customer_state', 'dominant_product_category']
global_mean      = df_train['is_late_delivery'].mean()
target_freq_maps = {}

for col in TARGET_FREQ_COLS:
    if col not in df_train.columns:
        continue
    freq_map             = df_train.groupby(col)['is_late_delivery'].mean()
    target_freq_maps[col] = freq_map
    enc_col              = col + '_target_enc'
    for df_s in [df_train, df_val, df_test]:
        df_s[enc_col] = df_s[col].map(freq_map).fillna(global_mean)
    print(f'Target-freq encoded: {col}  ->  {enc_col}')

# ── 8.2 One-hot encoding (drop_first) ─────────────────────────────────────
OHE_COLS       = ['customer_region', 'primary_payment_type']
ohe_categories = {}

for col in OHE_COLS:
    if col not in df_train.columns:
        continue
    cats               = sorted(df_train[col].dropna().unique())
    ohe_categories[col] = cats
    for df_s in [df_train, df_val, df_test]:
        for cat in cats[1:]:
            df_s[f'{col}__{cat}'] = (df_s[col] == cat).fillna(False).astype('int8')
    print(f'One-hot encoded: {col}  ->  {len(cats)-1} dummies  (dropped baseline: {cats[0]})')

# ── 8.3 Drop original categorical + identifier columns ─────────────────────
COLS_TO_DROP = (
    TARGET_FREQ_COLS + OHE_COLS
    + ['customer_zip_code_prefix', 'customer_city', 'customer_unique_id', 'order_id']
)
for df_s in [df_train, df_val, df_test]:
    df_s.drop(columns=[c for c in COLS_TO_DROP if c in df_s.columns], inplace=True)

print('\nOriginal categorical and identifier columns dropped.')


## 9. Separate Features & Target

In [ ]:
y_train = df_train['is_late_delivery'].reset_index(drop=True)
y_val   = df_val['is_late_delivery'].reset_index(drop=True)
y_test  = df_test['is_late_delivery'].reset_index(drop=True)

X_train = df_train.drop(columns=['is_late_delivery']).reset_index(drop=True)
X_val   = df_val.drop(columns=['is_late_delivery']).reset_index(drop=True)
X_test  = df_test.drop(columns=['is_late_delivery']).reset_index(drop=True)

# Enforce identical column order
X_val  = X_val[X_train.columns]
X_test = X_test[X_train.columns]

print(f'X_train : {X_train.shape}   y_train : {y_train.shape}')
print(f'X_val   : {X_val.shape}     y_val   : {y_val.shape}')
print(f'X_test  : {X_test.shape}    y_test  : {y_test.shape}')
print(f'\nFeature columns ({len(X_train.columns)}):')
for i, col in enumerate(X_train.columns, 1):
    print(f'  {i:>2}. {col}')


## 10. Final Leakage Re-Check (Post Feature Engineering)

In [ ]:
leakage_violations = [
    col for col in X_train.columns
    if any(frag in col.lower() for frag in PROHIBITED)
]

assert 'is_late_delivery' not in X_train.columns, 'Target in X_train!'
assert 'order_id'         not in X_train.columns, 'Identifier in X_train!'
assert not leakage_violations, f'Leakage columns: {leakage_violations}'
assert len(X_train) + len(X_val) + len(X_test) == len(merged), 'Row count mismatch!'
assert (X_val.columns == X_train.columns).all(),  'Column mismatch: train vs val'
assert (X_test.columns == X_train.columns).all(), 'Column mismatch: train vs test'

print('Leakage re-check PASSED  ✔')
print('  No prohibited column names in X sets.')
print('  Target and order_id not present.')
print('  Total row count preserved.')
print(f'  Final feature count : {len(X_train.columns)}')


## 11. Numeric Feature Scaling

`StandardScaler` is **fit on X_train only** and applied to X_val and X_test.
Boolean / flag / binary dummy columns are not scaled.


In [ ]:
DO_NOT_SCALE = (
    X_train.select_dtypes(include='int8').columns.tolist()
    + [c for c in X_train.columns
       if c.startswith(('customer_region__', 'primary_payment_type__'))]
)
SCALE_COLS = [
    c for c in X_train.select_dtypes(include=[np.number]).columns
    if c not in DO_NOT_SCALE
]

scaler = StandardScaler()
X_train[SCALE_COLS] = scaler.fit_transform(X_train[SCALE_COLS])
X_val[SCALE_COLS]   = scaler.transform(X_val[SCALE_COLS])
X_test[SCALE_COLS]  = scaler.transform(X_test[SCALE_COLS])

print(f'Scaled {len(SCALE_COLS)} numeric columns (fit on train only):')
for c in SCALE_COLS:
    print(f'  {c}')
print(f'\nNot scaled: {len(DO_NOT_SCALE)} flag/dummy/int8 columns.')


In [ ]:
# Scaling verification
scale_check = pd.DataFrame({
    'train_mean' : X_train[SCALE_COLS].mean().round(6),
    'train_std'  : X_train[SCALE_COLS].std().round(6),
    'val_mean'   : X_val[SCALE_COLS].mean().round(4),
    'test_mean'  : X_test[SCALE_COLS].mean().round(4),
})
display(Markdown('### 11.1 Scaling Verification (train mean ≈ 0, train std ≈ 1)'), scale_check)


**Interpretation.** Train means ≈ 0 and stds ≈ 1 confirm correct scaler application. Val/test means differ from 0 due to temporal distribution shift — expected and acceptable.

## 12. Final Dataset Summary

In [ ]:
summary_rows = []
for name, X, y in [('Train', X_train, y_train), ('Val', X_val, y_val), ('Test', X_test, y_test)]:
    summary_rows.append({
        'split'       : name,
        'rows'        : len(X),
        'features'    : X.shape[1],
        'late_count'  : int(y.sum()),
        'late_pct'    : round(y.mean()*100, 2),
        'any_null'    : int(X.isna().sum().sum()),
        'cols_ok'     : X.shape[1] == X_train.shape[1],
    })

summary_df = pd.DataFrame(summary_rows).set_index('split')
display(Markdown('### Final Modelling Dataset Summary'), summary_df)


## 13. Save Modelling-Ready Outputs

In [ ]:
# ── 13.1 Save CSV files ────────────────────────────────────────────────────
save_map = {
    'X_train.csv' : X_train,
    'X_val.csv'   : X_val,
    'X_test.csv'  : X_test,
    'y_train.csv' : y_train.to_frame(),
    'y_val.csv'   : y_val.to_frame(),
    'y_test.csv'  : y_test.to_frame(),
}
for filename, df_out in save_map.items():
    path = FINAL_DIR / filename
    df_out.to_csv(path, index=False, lineterminator='\n')
    print(f'Saved: data/final/{filename}  ({len(df_out):,} rows x {df_out.shape[1]} cols)')

# ── 13.2 Save scaler ───────────────────────────────────────────────────────
scaler_path = FINAL_DIR / 'scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'Saved: data/final/scaler.pkl')

# ── 13.3 Save feature names ────────────────────────────────────────────────
feature_path = FINAL_DIR / 'feature_names.txt'
feature_path.write_text('\n'.join(X_train.columns.tolist()))
print(f'Saved: data/final/feature_names.txt  ({len(X_train.columns)} features)')

# ── 13.4 Save target-frequency encoding maps ───────────────────────────────
enc_path = FINAL_DIR / 'target_freq_encodings.pkl'
with open(enc_path, 'wb') as f:
    pickle.dump(target_freq_maps, f)
print(f'Saved: data/final/target_freq_encodings.pkl')

print('\nAll outputs saved to data/final/  ✔')


In [ ]:
# ── 13.5 Reload validation ─────────────────────────────────────────────────
reload_rows = []
for filename, df_orig in save_map.items():
    reloaded = pd.read_csv(FINAL_DIR / filename)
    reload_rows.append({
        'file'       : filename,
        'rows_match' : len(reloaded) == len(df_orig),
        'cols_match' : reloaded.shape[1] == df_orig.shape[1],
        'rows'       : len(reloaded),
        'cols'       : reloaded.shape[1],
        'any_null'   : int(reloaded.isna().sum().sum()),
        'duplicates' : int(reloaded.duplicated().sum()),
    })

reload_df = pd.DataFrame(reload_rows).set_index('file')
assert reload_df['rows_match'].all(), 'Row count mismatch on reload!'
assert reload_df['cols_match'].all(), 'Column count mismatch on reload!'
display(Markdown('### 13.5 Reload Validation'), reload_df)
print('\nAll reloaded files match saved shape  ✔')


## 14. Findings & Decisions Summary

### 14.1 Merge Strategy
- **5 datasets merged** onto the `orders_clean` spine (96,470 rows) using validated left joins.
- `sellers_geo_clean` and `geolocation_zip_clean` loaded for audit only — their content  
  already flows through `customers_geo_clean` and `order_geo_features`.
- `customer_state` / `customer_region` conflict resolved by preferring `order_geo_features` versions.

### 14.2 Leakage Decisions
- `min/max_shipping_limit_date` → converted to `shipping_limit_lead_days`, then dropped.
- `order_purchase_timestamp` → used for split only, then dropped.
- `customer_id`, `order_id`, `customer_unique_id`, `customer_city`, `customer_zip_code_prefix` → identifiers, dropped.

### 14.3 Missing Value Strategy (all fit on train)
- Distance columns (1.3%): **train-median imputation**
- Seller flag columns (0.8%): **fill False** (no seller bridge treated as unknown)
- Product columns (0.8%): **fill 0** (no item-bridge record)
- `dominant_product_category` (rare): **fill 'other'**
- `shipping_limit_lead_days`: **train-median imputation**

### 14.4 Feature Engineering (8 new cross-dataset features)
`freight_to_price_ratio`, `weight_per_item`, `volume_per_item`, `is_multi_seller`,  
`is_cross_state`, `distance_x_items`, `freight_per_km`, `purchase_quarter`

### 14.5 Encoding & Scaling
- Target-frequency encoding (train-fit): `customer_state`, `dominant_product_category`
- One-hot encoding (drop-first): `customer_region`, `primary_payment_type`
- StandardScaler (train-fit): all continuous numeric columns

### 14.6 Split
- **Chronological 70/15/15** — not random — to prevent temporal leakage.
- Train: ≤ Apr 2018 | Val: Apr–Jun 2018 | Test: > Jun 2018

### 14.7 Class Imbalance Note
Target is 11.3:1 imbalanced. SMOTE, class-weight tuning, or threshold adjustment  
should be applied during modelling (Notebook 06+), not here.
